# SST-5: Stanford Sentiment Treebank (5 classes)
# ESN Bayesian Optimization (``ax-platform``) + Simulations

## Librairies

In [1]:
# !pip install transformers==4.8.2
# !pip install datasets==1.7.0
# !pip install ax-platform==0.1.20

In [2]:
import os
import sys
sys.path.insert(0, os.path.abspath("../..")) # comment this if library is pip installed

In [3]:
import io
import re
import pickle
from timeit import default_timer as timer

from tqdm.notebook import tqdm

import numpy as np
import random

import torch
import torch.nn as nn
import torch.optim as optim

from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import AutoTokenizer
from transformers import BertModel
from transformers.data.data_collator import DataCollatorWithPadding

from ax import optimize
from ax.plot.contour import plot_contour
from ax.plot.trace import optimization_trace_single_method
from ax.service.managed_loop import optimize
from ax.utils.notebook.plotting import render, init_notebook_plotting

import esntorch.core.reservoir as res
import esntorch.core.learning_algo as la
import esntorch.core.pooling_strategy as ps
import esntorch.core.esn as esn

In [4]:
%config Completer.use_jedi = False
%load_ext autoreload
%autoreload 2

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [6]:
# SEED = 42

## Global variables

In [7]:
pooling_strategy = 'last' # 'mean' or 'last'

In [8]:
RESULTS_PATH = 'Results/ESN_v2' # put your path here
CACHE_DIR = '/huggingface_datasets' # put your path here
PARAMS_FILE = 'sst-5_params_uniform_'+pooling_strategy+'-pooling.pkl'
RESULTS_FILE = 'sst-5_results_esn_uniform_'+pooling_strategy+'-pooling_FINAL.pkl'

In [9]:
RESULTS_FILE

'sst-5_results_esn_uniform_last-pooling_FINAL.pkl'

## Dataset

In [10]:
# rename correct column as 'labels': depends on the dataset you load

def tokenize(sample):
    """Tokenize sample"""
    
    sample = tokenizer(sample['text'], truncation=True, padding=False, return_length=True)
    
    return sample
    
def load_and_enrich_dataset(dataset_name, split, cache_dir):
    """
    Load dataset from the datasets library of HuggingFace.
    Tokenize and add length.
    """
    
    # Load dataset
    dataset = load_dataset(dataset_name, split=split, cache_dir=CACHE_DIR)
    
    # Rename label column for tokenization purposes (use 'label-fine' for fine-grained labels)
    dataset = dataset.rename_column('label', 'labels')
    
    # Tokenize data
    dataset = dataset.map(tokenize, batched=True)
    dataset = dataset.rename_column('length', 'lengths')
    dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels', 'lengths'])
    
    return dataset

In [11]:
# tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# train set
# full_train_dataset = load_and_enrich_dataset('SetFit/sst5', split='train', cache_dir=CACHE_DIR).sort("lengths")

# mini train and val sets: 70%/30% of 30%
# XXX here we increase the train_size split since small dataset
train_dataset = load_and_enrich_dataset('SetFit/sst5', split='train', cache_dir=CACHE_DIR).sort("lengths")
val_dataset = load_and_enrich_dataset('SetFit/sst5', split='validation', cache_dir=CACHE_DIR).sort("lengths")
full_train_dataset = concatenate_datasets([train_dataset, val_dataset]).sort("lengths")

# test set
test_dataset = load_and_enrich_dataset('SetFit/sst5', split='test', cache_dir=CACHE_DIR).sort("lengths")

dataset_d = {
    'full_train': full_train_dataset,
    'train': train_dataset,
    'val': val_dataset,
    'test': test_dataset
    }

# dataloaders
dataloader_d = {}
for k, v in dataset_d.items():
    dataloader_d[k] = torch.utils.data.DataLoader(v, batch_size=256, collate_fn=DataCollatorWithPadding(tokenizer))

Repo card metadata block was not found. Setting CardData to empty.
Repo card metadata block was not found. Setting CardData to empty.


Map:   0%|          | 0/1101 [00:00<?, ? examples/s]

Repo card metadata block was not found. Setting CardData to empty.


In [12]:
dataset_d

{'full_train': Dataset({
     features: ['text', 'labels', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask', 'lengths'],
     num_rows: 9645
 }),
 'train': Dataset({
     features: ['text', 'labels', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask', 'lengths'],
     num_rows: 8544
 }),
 'val': Dataset({
     features: ['text', 'labels', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask', 'lengths'],
     num_rows: 1101
 }),
 'test': Dataset({
     features: ['text', 'labels', 'label_text', 'input_ids', 'token_type_ids', 'attention_mask', 'lengths'],
     num_rows: 2210
 })}

In [13]:
print("Computing BERT embeddings once...")

bert = BertModel.from_pretrained("bert-base-uncased").to(device)
bert.eval()

def compute_full_embeddings(dataloader):
    """
    Devuelve:
      - all_embeddings: lista de tensores (seq_len_i, 768) SIN padding
      - all_labels:     tensor de enteros
      - all_lengths:    tensor de enteros
    """
    all_embeddings = []
    all_labels = []
    all_lengths = []

    with torch.no_grad():
        for batch in tqdm(dataloader):
            labels   = batch["labels"]            # (B,)
            lengths  = batch["lengths"]           # (B,)

            inputs = {
                "input_ids":      batch["input_ids"].to(device),
                "attention_mask": batch["attention_mask"].to(device),
            }
            outputs = bert(**inputs)
            hidden = outputs.last_hidden_state    # (B, T_pad, 768)

            # Recortamos el padding de cada muestra individualmente
            for i in range(hidden.size(0)):
                l = lengths[i].item()
                all_embeddings.append(hidden[i, :l, :].cpu())  # (l, 768)

            all_labels.append(labels)
            all_lengths.append(lengths)

            del outputs
            torch.cuda.empty_cache()

    return all_embeddings, torch.cat(all_labels), torch.cat(all_lengths)

# Precomputamos para cada split
emb_full_train_X, emb_full_train_y, emb_full_train_l = compute_full_embeddings(dataloader_d["full_train"])
emb_train_X,      emb_train_y,      emb_train_l      = compute_full_embeddings(dataloader_d["train"])
emb_val_X,        emb_val_y,        emb_val_l        = compute_full_embeddings(dataloader_d["val"])
emb_test_X,       emb_test_y,       emb_test_l       = compute_full_embeddings(dataloader_d["test"])

# Liberamos BERT de memoria — ya no lo necesitamos
del bert
torch.cuda.empty_cache()

# Guardado opcional para no recomputar entre sesiones
torch.save(
    {
        "full_train": (emb_full_train_X, emb_full_train_y, emb_full_train_l),
        "train":      (emb_train_X,      emb_train_y,      emb_train_l),
        "val":        (emb_val_X,        emb_val_y,        emb_val_l),
        "test":       (emb_test_X,       emb_test_y,       emb_test_l),
    },
    "sst-5_full_embeddings.pt"
)

Computing BERT embeddings once...


  0%|          | 0/38 [00:00<?, ?it/s]

  0%|          | 0/34 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

In [14]:
from torch.utils.data import Dataset

class EmbeddingDataset(Dataset):
    """Dataset de embeddings BERT precomputados."""
    def __init__(self, embeddings, labels, lengths):
        self.embeddings = embeddings  # lista de tensores (l_i, 768)
        self.labels     = labels      # tensor (N,)
        self.lengths    = lengths     # tensor (N,)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "embeddings": self.embeddings[idx],   # (l_i, 768)
            "labels":     self.labels[idx],
            "lengths":    self.lengths[idx],
            "input_ids":  self.lengths[idx],      # ← dummy escalar; ESN.warm_up
                                                  #   hará unsqueeze(0) → shape [1]
        }


def embedding_collate_fn(batch):
    """Hace padding de los embeddings al máximo largo del batch."""
    embeddings = [item["embeddings"] for item in batch]
    labels     = torch.stack([item["labels"]  for item in batch])
    lengths    = torch.stack([item["lengths"] for item in batch])

    max_len = max(e.size(0) for e in embeddings)
    hidden_dim = embeddings[0].size(1)  # 768
    padded = torch.zeros(len(embeddings), max_len, hidden_dim)
    for i, e in enumerate(embeddings):
        padded[i, :e.size(0), :] = e

    return {
        "embeddings": padded,   # (B, T_max, 768)
        "labels":     labels,
        "lengths":    lengths,
        "input_ids":  torch.zeros(len(batch), 1, dtype=torch.long),  # [B, 1] dummy
    }


# Construimos los datasets y dataloaders de embeddings
emb_dataset_d = {
    "full_train": EmbeddingDataset(emb_full_train_X, emb_full_train_y, emb_full_train_l),
    "train":      EmbeddingDataset(emb_train_X,      emb_train_y,      emb_train_l),
    "val":        EmbeddingDataset(emb_val_X,         emb_val_y,       emb_val_l),
    "test":       EmbeddingDataset(emb_test_X,        emb_test_y,      emb_test_l),
}

emb_dataloader_d = {
    k: torch.utils.data.DataLoader(
            v, batch_size=256, collate_fn=embedding_collate_fn
        )
    for k, v in emb_dataset_d.items()
}

In [15]:
class PrecomputedEmbedding(nn.Module):
    """
    Módulo que sustituye a BERT dentro del ESN.
    En lugar de calcular embeddings, simplemente extrae el tensor
    precomputado del batch y lo transpone al formato que espera _embed.
    """
    def forward(self, batch):
        # batch["embeddings"]: [Batch_Size, Length, Hidden_Size]  →  devuelve [Length, Batch_Size, Hidden_Size]
        return batch["embeddings"].permute(1, 0, 2)

## Optimization

- Parameters

In [16]:
ESN_PARAMS = {
            'embedding': None, #'bert-base-uncased', # TEXT.vocab.vectors,
            'distribution' : 'uniform',               # uniform, gaussian
            'input_dim' : 768,                        # dim of encoding!
            'reservoir_dim' : None,
            'bias_scaling' : 0.0,
            'sparsity' : None,
            'spectral_radius' : None,
            'leaking_rate': None,
            'activation_function' : 'tanh',
            'input_scaling' : None,
            'mean' : 0.0,
            'std' : 1.0,
            'learning_algo' : None,
            'criterion' : None,
            'optimizer' : None,
            'pooling_strategy' : pooling_strategy,
            'lexicon' : None,
            'bidirectional' : False, # False
            'mode' : None,
            'device' : device,
            'seed' : None
             }

In [17]:
def plug_parameters(args_d):
    
    params_d = ESN_PARAMS.copy()
    
    for k, v in args_d.items():
        if k in params_d.keys():
            params_d[k] = v

    # params_d['reservoir_dim'] = int(params_d['reservoir_dim'])
    # params_d['spectral_radius'] = float(params_d['spectral_radius'])
    # params_d['leaking_rate'] = float(params_d['leaking_rate'])
    # params_d['input_scaling'] = float(params_d['input_scaling'])
    # params_d['sparsity'] = float(params_d['sparsity'])

    params_d['dim'] = int(args_d['reservoir_dim'])
            
    return params_d

In [18]:
# args_d = {'reservoir_dim': 1000, 
#      'spectral_radius': 0.9, 
#      'leaking_rate': 0.5, 
#      'input_scaling': 0.1, 
#      'sparsity': 0.99, 
#      'activation_function': 'tanh', 
#      'alpha': 1.0}

# d = plug_parameters(args_d)

# d

- Fitness function

In [19]:
def warm_up(ESN, emb_dataset_d):
    """Warm up usando embeddings precomputados."""
    nb_sentences = 10
    subset = torch.utils.data.Subset(emb_dataset_d["train"], range(nb_sentences))
    ESN.warm_up(subset) # ESN.warm_up itera, hace unsqueeze y BatchEncoding internamente
    
    # """Warm up the ESN."""
    
    # nb_sentences = 10
    
    # for i in range(nb_sentences):
    #     # Seleccionamos una muestra del EmbeddingDataset
    #     sample = emb_dataset_d["train"][i]
    #     # warm_up espera un batch con las claves adecuadas
    #     batch = embedding_collate_fn([sample])
    #     ESN.warm_up([batch])

        # sentence = dataset_d["train"].select([i])
        # dataloader_tmp = torch.utils.data.DataLoader(sentence, 
        #                                              batch_size=1, 
        #                                              collate_fn=DataCollatorWithPadding(tokenizer))  

        # ESN.warm_up(sentence)

In [20]:
def fitness(reservoir_dim=1000,
            spectral_radius=0.9,
            leaking_rate=0.5,
            input_scaling=0.1, 
            sparsity=0.99,
            activation_function='tanh',
            alpha=1.0,
            emb_dataset_d=emb_dataset_d, # dataset_d, 
            emb_dataloader_d=emb_dataloader_d, # dataloader_d, 
            seed_l=random.sample(range(1, 9999999), 3), # 3 seeds
            return_test_acc=False):
    
    # XXX
    if return_test_acc:
        seed_l=random.sample(range(1, 9999999999), 10) # 10 seeds
    elif not return_test_acc:
        seed_l=random.sample(range(1, 9999999), 3) # 3 seeds
    print('SEEDS:', seed_l)
    # XXX
    
    # create esn parameters
    args_d = locals()
    esn_params = plug_parameters(args_d)
    print("\nParámetros ESN:\n", esn_params)
    
    # BO mode: param optimization (esn_params['mode'] = 'recurrent_layer')
    if not return_test_acc:
        print("\nOPTIMIZACIÓN BAYESIANA\n")

        acc_l = []
    
        for seed in tqdm(seed_l):
            
            # model
            esn_params['mode'] = 'recurrent_layer'
            esn_params['seed'] = seed
            ESN = esn.EchoStateNetwork(**esn_params)
            ESN.layer.embedding = PrecomputedEmbedding()  # ← inyección post-construcción
            ESN.learning_algo = la.RidgeRegression(alpha = alpha)
            ESN = ESN.to(device)

            # warm up
            warm_up(ESN, emb_dataset_d)
            
            # train
            LOSS = ESN.fit(emb_dataloader_d["train"])                    # mini train set
            acc = ESN.predict(emb_dataloader_d["val"], verbose=False)[1] # mini val set
            acc_l.append(acc)
            
            # clean objects
            del ESN.learning_algo
            del ESN.criterion
            del ESN.pooling_strategy
            del ESN
            torch.cuda.empty_cache()
            
        # results
        return np.mean(acc_l) 
          
    # prediction mode: once params have been optimized
    elif return_test_acc:
        print("\nMODO PREDICCIÓN\n")
        print('return_test_acc mode')
        
        # esn
        acc_esn_l = []
        time_esn_l = []
        # custom baseline
        acc_bs_l = []
        time_bs_l = []
        
        for seed in tqdm(seed_l):
            
            for mode in tqdm(['recurrent_layer', 'linear_layer']):
            
                # model
                esn_params['mode'] = mode
                esn_params['seed'] = seed
                ESN = esn.EchoStateNetwork(**esn_params)
                ESN.layer.embedding = PrecomputedEmbedding()  # ← inyección post-construcción
                ESN.learning_algo = la.RidgeRegression(alpha = alpha)
                ESN = ESN.to(device)
                print(ESN.layer.dim)

                # warm up solo para la capa recurrente
                if mode == 'recurrent_layer':
                    warm_up(ESN, emb_dataset_d)

                # train
                t0 = timer()
                LOSS = ESN.fit(emb_dataloader_d["full_train"])                # full train set
                t1 = timer()
                acc = ESN.predict(emb_dataloader_d["test"], verbose=False)[1] # full test set
                
                # clean objects
                del ESN.learning_algo
                del ESN.criterion
                del ESN.pooling_strategy
                del ESN
                torch.cuda.empty_cache()
                
                # results
                if mode == 'recurrent_layer':
                    time_esn_l.append(t1 - t0)
                    acc_esn_l.append(acc)
                elif mode == 'linear_layer':
                    time_bs_l.append(t1 - t0)
                    acc_bs_l.append(acc)
            
        results_d = {}
        results_d['recurrent_layer'] = [np.mean(acc_esn_l), np.std(acc_esn_l), np.mean(time_esn_l), np.std(time_esn_l)]
        results_d['linear_layer'] = [np.mean(acc_bs_l), np.std(acc_bs_l), np.mean(time_bs_l), np.std(time_bs_l)]
            
        return results_d

In [21]:
# %%time

# fitness(
#         reservoir_dim=1000,
#         spectral_radius=0.9,
#         leaking_rate=0.5,
#         input_scaling=0.1, 
#         sparsity=0.99,
#         activation_function='tanh',
#         alpha=1.0,
#         dataset_d=dataset_d, 
#         dataloader_d=dataloader_d,
#         seed_l=[1234321, 7],
#         return_test_acc=True
# )

In [22]:
def wrapped_fitness(d, return_test_acc=False):
    
    return fitness(reservoir_dim=d['reservoir_dim'], # will be in the loop 
                   spectral_radius=d['spectral_radius'],
                   leaking_rate=d['leaking_rate'],
                   input_scaling=d['input_scaling'],
                   sparsity=d['sparsity'],
                   activation_function=d['activation_function'],
                   alpha=d['alpha'],
                   emb_dataset_d=emb_dataset_d, # dataset_d,
                   emb_dataloader_d=emb_dataloader_d, # dataloader_d,
                   return_test_acc=return_test_acc)

In [23]:
# *** WARNING *** DO NO EXECUTE NEXT CELLS IF BIDIRECTIONAL MODE (OPTIM ALREADY DONE)

- Optimization

In [24]:
best_params_d = {}

for res_dim in tqdm([500, 1000, 2000, 3000]):
    
    best_parameters, best_values, experiment, model = optimize(
            parameters=[
              {
                "name": "reservoir_dim",
                "value_type": "int",
                "type": "fixed",
                "value": res_dim,
              },
              {
                "name": "activation_function",
                "value_type": "str",
                "type": "fixed",
                "value": 'tanh',
              },
              {
                "name": "spectral_radius",
                "value_type": "float",
                "type": "choice",
                "values": [0.5, 1.0, 1.5],
              },
              {
                "name": "leaking_rate",
                "value_type": "float",
                "type": "choice",
                "values": [0.1, 0.5, 0.9],
              },
              {
                "name": "input_scaling",
                "value_type": "float",
                "type": "choice",
                "values": [0.1, 1.0],
              },
              {
                "name": "sparsity",
                "value_type": "float",
                "type": "choice",
                "values": [0.0, 0.5, 0.9],
              },
              {
                "name": "alpha",
                "value_type": "float",
                "type": "choice",
                "values": [0.1, 1.0, 10.0, 100.0],
              }
            ],
            # Booth function
            evaluation_function = wrapped_fitness,
            minimize = False,
            objective_name = 'val_accuracy',
            total_trials = 30 # nb iterations here 20 original
        )
    
    # results
    best_params_d[res_dim] = {}
    best_params_d[res_dim]['best_parameters'] = best_parameters
    best_params_d[res_dim]['best_values'] = best_values
    best_params_d[res_dim]['experiment'] = experiment
    # best_params_d[res_dim]['model'] = model

  0%|          | 0/4 [00:00<?, ?it/s]

c:\Users\AJYH\AppData\Local\Programs\Python\Python38\lib\site-packages\ax\core\parameter.py:517: UserWarning: `is_ordered` is not specified for `ChoiceParameter` "spectral_radius". Defaulting to `True` for parameters of `ParameterType` FLOAT. To override this behavior (or avoid this warning), specify `is_ordered` during `ChoiceParameter` construction.
  warn(
c:\Users\AJYH\AppData\Local\Programs\Python\Python38\lib\site-packages\ax\core\parameter.py:517: UserWarning: `sort_values` is not specified for `ChoiceParameter` "spectral_radius". Defaulting to `True` for parameters of `ParameterType` FLOAT. To override this behavior (or avoid this warning), specify `sort_values` during `ChoiceParameter` construction.
  warn(
c:\Users\AJYH\AppData\Local\Programs\Python\Python38\lib\site-packages\ax\core\parameter.py:517: UserWarning: `is_ordered` is not specified for `ChoiceParameter` "leaking_rate". Defaulting to `True` for parameters of `ParameterType` FLOAT. To override this behavior (or avoi

SEEDS: [1408090, 9496798, 1606619]

Parámetros ESN:
 {'embedding': None, 'distribution': 'uniform', 'input_dim': 768, 'reservoir_dim': 500, 'bias_scaling': 0.0, 'sparsity': 0.5, 'spectral_radius': 0.5, 'leaking_rate': 0.9, 'activation_function': 'tanh', 'input_scaling': 1.0, 'mean': 0.0, 'std': 1.0, 'learning_algo': None, 'criterion': None, 'optimizer': None, 'pooling_strategy': 'last', 'lexicon': None, 'bidirectional': False, 'mode': None, 'device': device(type='cpu'), 'seed': None, 'dim': 500}

OPTIMIZACIÓN BAYESIANA



  0%|          | 0/3 [00:00<?, ?it/s]

Computing closed-form solution...


...

  0%|          | 0/5 [00:00<?, ?it/s]

Computing closed-form solution...


  0%|          | 0/34 [00:00<?, ?it/s]


Training complete.


  0%|          | 0/5 [00:00<?, ?it/s]

In [25]:
best_params_d

{500: {'best_parameters': {'spectral_radius': 0.5,
   'leaking_rate': 0.1,
   'input_scaling': 0.1,
   'sparsity': 0.0,
   'alpha': 0.1,
   'reservoir_dim': 500,
   'activation_function': 'tanh'},
  'best_values': ({'val_accuracy': 42.16835848308176},
   {'val_accuracy': {'val_accuracy': 0.014058428254874378}}),
  'experiment': Experiment(None)},
 1000: {'best_parameters': {'spectral_radius': 0.5,
   'leaking_rate': 0.1,
   'input_scaling': 0.1,
   'sparsity': 0.5,
   'alpha': 1.0,
   'reservoir_dim': 1000,
   'activation_function': 'tanh'},
  'best_values': ({'val_accuracy': 41.290076925636455},
   {'val_accuracy': {'val_accuracy': 0.011788896131662096}}),
  'experiment': Experiment(None)},
 2000: {'best_parameters': {'spectral_radius': 0.5,
   'leaking_rate': 0.1,
   'input_scaling': 1.0,
   'sparsity': 0.9,
   'alpha': 100.0,
   'reservoir_dim': 2000,
   'activation_function': 'tanh'},
  'best_values': ({'val_accuracy': 41.9326748289072},
   {'val_accuracy': {'val_accuracy': 0.02147

In [26]:
# Crear la carpeta si no existe
os.makedirs(RESULTS_PATH, exist_ok=True)

# save parameters

with open(os.path.join(RESULTS_PATH, PARAMS_FILE), 'wb') as fh:
    pickle.dump(best_params_d, fh)

In [27]:
# load params

with open(os.path.join(RESULTS_PATH, PARAMS_FILE), 'rb') as fh:
    best_params_d = pickle.load(fh)
    
PARAMS_FILE

'sst-5_params_uniform_last-pooling.pkl'

In [28]:
best_params_d

{500: {'best_parameters': {'spectral_radius': 0.5,
   'leaking_rate': 0.1,
   'input_scaling': 0.1,
   'sparsity': 0.0,
   'alpha': 0.1,
   'reservoir_dim': 500,
   'activation_function': 'tanh'},
  'best_values': ({'val_accuracy': 42.16835848308176},
   {'val_accuracy': {'val_accuracy': 0.014058428254874378}}),
  'experiment': Experiment(None)},
 1000: {'best_parameters': {'spectral_radius': 0.5,
   'leaking_rate': 0.1,
   'input_scaling': 0.1,
   'sparsity': 0.5,
   'alpha': 1.0,
   'reservoir_dim': 1000,
   'activation_function': 'tanh'},
  'best_values': ({'val_accuracy': 41.290076925636455},
   {'val_accuracy': {'val_accuracy': 0.011788896131662096}}),
  'experiment': Experiment(None)},
 2000: {'best_parameters': {'spectral_radius': 0.5,
   'leaking_rate': 0.1,
   'input_scaling': 1.0,
   'sparsity': 0.9,
   'alpha': 100.0,
   'reservoir_dim': 2000,
   'activation_function': 'tanh'},
  'best_values': ({'val_accuracy': 41.9326748289072},
   {'val_accuracy': {'val_accuracy': 0.02147

## Simulations
- Using best paramameters

In [29]:
# results

results_d = {}

for res_dim in tqdm([500, 1000, 2000, 3000]):
    
    best_parameters = best_params_d[res_dim]['best_parameters']
    d = wrapped_fitness(best_parameters, return_test_acc=True)
    results_d[res_dim] = d
    print("Experiment finished.")

  0%|          | 0/4 [00:00<?, ?it/s]

SEEDS: [7017487713, 9687496293, 9995759255, 4588994942, 4540788310, 3761393623, 2773501780, 8110211349, 278824532, 133818701]

Parámetros ESN:
 {'embedding': None, 'distribution': 'uniform', 'input_dim': 768, 'reservoir_dim': 500, 'bias_scaling': 0.0, 'sparsity': 0.0, 'spectral_radius': 0.5, 'leaking_rate': 0.1, 'activation_function': 'tanh', 'input_scaling': 0.1, 'mean': 0.0, 'std': 1.0, 'learning_algo': None, 'criterion': None, 'optimizer': None, 'pooling_strategy': 'last', 'lexicon': None, 'bidirectional': False, 'mode': None, 'device': device(type='cpu'), 'seed': None, 'dim': 500}

MODO PREDICCIÓN

return_test_acc mode


  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

500
Computing closed-form solution...


...

3000
Computing closed-form solution...


  0%|          | 0/38 [00:00<?, ?it/s]


Training complete.


  0%|          | 0/9 [00:00<?, ?it/s]

Experiment finished.


In [30]:
results_d

{500: {'recurrent_layer': [43.64253393665159,
   0.49246487376342757,
   6.780950709999888,
   1.9831949529680926],
  'linear_layer': [42.7420814479638,
   0.4641925286237894,
   5.20850991999996,
   1.9696637417339051]},
 1000: {'recurrent_layer': [43.004524886877824,
   0.7990494137427738,
   13.285103609999897,
   1.5069919140146077],
  'linear_layer': [42.502262443438916,
   0.7492890237231891,
   8.036543269999857,
   0.5671441167866313]},
 2000: {'recurrent_layer': [43.457013574660635,
   0.7410875144822886,
   33.79968265000025,
   1.1460206860129012],
  'linear_layer': [39.303167420814475,
   0.6763759978662371,
   14.93193870000032,
   0.8087077403796115]},
 3000: {'recurrent_layer': [42.87782805429865,
   0.6383779941555981,
   66.43203391000006,
   2.9987988617845858],
  'linear_layer': [45.23529411764706,
   0.492797367597418,
   28.546889829999962,
   4.212981091102966]}}

In [31]:
RESULTS_FILE

'sst-5_results_esn_uniform_last-pooling_FINAL.pkl'

In [32]:
# save results

with open(os.path.join(RESULTS_PATH, RESULTS_FILE), 'wb') as fh:
    pickle.dump(results_d, fh)

In [33]:
# # load results
# with open(os.path.join(RESULTS_PATH, RESULTS_FILE), 'rb') as fh:
#     results_d = pickle.load(fh)